# 04d — EXP_04: RAGAS Evaluation (Claude Sonnet 4.6 judge — all 5 metrics)

Scores `results/exp_04_hybrid_rag__golden_234/predictions.jsonl` against the 234-row golden subset.

**Headline comparison numbers** to fill in:

| Architecture | Acuuracy (test_1273) | Faithfulness | Hallucination Rate | Context Precision | Context Recall |
|---|---:|---:|---:|---:|---:|
| EXP_01 No-RAG | 0.7738 | n/a | n/a | n/a | n/a |
| EXP_02 Naive Dense | 0.7573 | 0.1308 | 0.8957 | 0.3285 | 0.4124 |
| EXP_03 Sparse BM25 | 0.7581 | (pending) | (pending) | (pending) | (pending) |
| **EXP_04 Hybrid** | **??** | **??** | **??** | **??** | **??** |

**Falsifiable hypothesis** (anchored in [`docs/output_notes/04b_exp02_output.md` §4](../docs/output_notes/04b_exp02_output.md)): *Hybrid Context Precision will be ≥ 0.50 by combining dense + sparse top-k. Falsified if Hybrid Context Precision ≤ Naive's 0.33.*

Stages: smoke 10 (~$0.50, ~1 min) → full 234 (~$11–12, ~10–20 min) → NaN rescore if needed.

## 1. Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

from src.eval.ragas_eval import score_predictions

for k in ["ANTHROPIC_API_KEY"]:
    print(f"{k}: {'\u2713 present' if os.environ.get(k) else '\u2717 MISSING'}")
print("Repo root:", REPO_ROOT)

## 2. Configuration

In [ ]:
PRED_DIR = REPO_ROOT / "results" / "exp_04_hybrid_rag__golden_234"
GOLDEN_PATH = REPO_ROOT / "data" / "processed" / "golden_ragas_300.jsonl"
CHUNKS_PATH = REPO_ROOT / "data" / "processed" / "chunks.parquet"

RUN_SMOKE = True
SMOKE_N = 10
RUN_FULL = False
RUN_RESCORE_NANS = False
OVERWRITE_SCORES = False

print("Predictions:", PRED_DIR)
print("Golden     :", GOLDEN_PATH)
print("Chunks     :", CHUNKS_PATH)

assert PRED_DIR.exists(), (
    f"{PRED_DIR} doesn't exist — run 04d_exp04_hybrid_rag.ipynb Stage B first."
)

## 3. Stage A — Smoke (10 rows × 5 metrics · ~1 min · ~$0.50)

In [ ]:
import pandas as pd
import shutil

if RUN_SMOKE:
    smoke_dir = PRED_DIR.parent / "exp_04_hybrid_rag__golden_234_ragas_smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    for fname in ("predictions.jsonl", "retrieval.jsonl", "summary.json"):
        shutil.copy2(PRED_DIR / fname, smoke_dir / fname)

    smoke_summary = score_predictions(
        predictions_dir=smoke_dir,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        n_smoke=SMOKE_N,
        overwrite=OVERWRITE_SCORES,
    )
    print('\n=== Stage A smoke summary (10 rows × 5 metrics) ===')
    for k, v in smoke_summary.items():
        if k.startswith(("RAGAS_", "Answer_", "ragas_")):
            print(f"  {k:30s} : {v}")
    smoke_scores = pd.read_csv(smoke_dir / "ragas_scores.csv")
    cols = [c for c in smoke_scores.columns if c in ("question_id", "_is_correct", "faithfulness", "context_precision", "context_recall", "answer_relevancy", "answer_correctness")]
    print("\n=== Per-row scores (smoke 10) ===")
    print(smoke_scores[cols].to_string(index=False))
else:
    print("RUN_SMOKE = False — skipped.")

**Acceptance gates (Stage A)** — flip `RUN_FULL = True` only when all five pass: (1) no errors, (2) all 5 metric cols present, (3) NaN rate < 30 %, (4) Faithfulness ≥ 0 on at least some rows (EXP_02 had 0.13; EXP_04 should be similar or higher), (5) `_is_correct=True` rows average higher Answer Correctness than `_is_correct=False`.

## 4. Stage B — Full 234 (~10–20 min · ~$11–12)

In [ ]:
if RUN_FULL:
    full_summary = score_predictions(
        predictions_dir=PRED_DIR,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        n_smoke=None,
        overwrite=OVERWRITE_SCORES,
    )
    print("\n=== Stage B full summary (234 rows × 5 metrics) ===")
    for k, v in full_summary.items():
        if k.startswith(("RAGAS_", "Answer_", "ragas_")) or k == "Acuuracy":
            print(f"  {k:30s} : {v}")
else:
    print("RUN_FULL = False")

## 5. Stage C — NaN rescore

In [ ]:
if RUN_RESCORE_NANS:
    rescored_summary = score_predictions(
        predictions_dir=PRED_DIR,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        rescore_nans=True,
    )
    df = pd.read_csv(PRED_DIR / "ragas_scores.csv")
    for col in ("faithfulness", "context_precision", "context_recall", "answer_relevancy", "answer_correctness"):
        if col in df.columns:
            n_nan = pd.to_numeric(df[col], errors="coerce").isna().sum()
            print(f"  {col:25s}: {n_nan} NaN remaining of {len(df)}")
else:
    print("RUN_RESCORE_NANS = False")

## 6. Inspect — does Hybrid clear the Context Precision bar?

In [ ]:
scores_path = PRED_DIR / "ragas_scores.csv"
if scores_path.exists():
    df = pd.read_csv(scores_path)
    print(f"\nrows: {len(df)}")
    print('\n--- EXP_04 Hybrid vs EXP_02 Naive Dense (golden 234) ---')
    e2 = json.loads((REPO_ROOT / 'results' / 'exp_02_naive_rag__golden_234' / 'summary.json').read_text())
    for col, e2_key in [
        ('faithfulness', 'RAGAS_Faithfulness'),
        ('context_precision', 'RAGAS_Context_Precision'),
        ('context_recall', 'RAGAS_Context_Recall'),
        ('answer_relevancy', 'RAGAS_Answer_Relevance'),
        ('answer_correctness', 'Answer_Correctness'),
    ]:
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors='coerce')
        e4_mean = s.mean()
        e2_mean = e2.get(e2_key)
        delta = (e4_mean - e2_mean) if isinstance(e2_mean, (int, float)) else None
        delta_s = f'{delta*100:+.2f} pp' if delta is not None else 'n/a'
        print(f'  {col:25s}: EXP_04={e4_mean:.4f}  EXP_02={e2_mean if e2_mean else "--":>8}  Δ={delta_s}  NaN={s.isna().sum()}')
    cp = pd.to_numeric(df['context_precision'], errors='coerce').mean() if 'context_precision' in df.columns else None
    if cp is not None:
        verdict = '\u2713 SUPPORTED' if cp >= 0.50 else ('partial — ≥ 0.33' if cp > 0.33 else '\u2717 FALSIFIED')
        print(f'\n  Hypothesis: Hybrid Context Precision ≥ 0.50? → {verdict}  (got {cp:.4f})')
else:
    print("No ragas_scores.csv yet — run Stage B first.")

---

**Next.** EXP_05 Multi-Hop RAG — already built ([`04e_exp05_multi_hop_rag.ipynb`](04e_exp05_multi_hop_rag.ipynb) + [`04e_exp05_ragas.ipynb`](04e_exp05_ragas.ipynb)).